# Notebook de creación de features

In [1]:
from configuraciones import *
import Utils.map_loader as ml

## Features del Modelo nnELM

1. Información Bottom-Up
2. Similitud del Objetivo (Target Similarity)
3. Evidencia Visual
4. Competencia Foveal-Periférica
5. Probabilidad Posterior
6. Entropía del Prior
7. Entropía Global del Posterior
8. Ganancia de Información
9. Gap de Entropía con el Óptimo
10. Distancia con la Decisión del Modelo
11. Competencia en Matriz de Entropías (Ratio)
12. Competencia en Matriz de Entropías (Densidad de Candidatos)

## Estructura de los resultados

Guardo las features en un JSON en Features/Imagen/Sujeto. Es decir, para cada imagen y cada sujeto tengo un JSON que me dice, para cada fijacion, el valor de cada una de las features

In [11]:
FEATURES_PATH = '../Features'

def cargar_o_crear_json(path: str, defaults: dict) -> dict:
    """Carga el JSON si existe, si no lo crea con los defaults."""
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return defaults

In [2]:
# Lista de sujetos
path_sujetos = '../Human_Maps/HSUBANOTT'
sujetos = sorted(os.listdir(path_sujetos))
print('Sujetos disponibles:', sujetos)
print('Número de sujetos:', len(sujetos))

Sujetos disponibles: ['S101', 'S102', 'S103', 'S105', 'S106', 'S107', 'S108', 'S109', 'S110', 'S111', 'S112', 'S113', 'S114', 'S115', 'S116', 'S117', 'S118', 'S119', 'et_117969', 'et_123082_2', 'et_179678', 'et_200877', 'et_244108', 'et_244389', 'et_248301', 'et_286470', 'et_305138', 'et_370500', 'et_373490', 'et_389622', 'et_400297', 'et_419760', 'et_501896', 'et_533569', 'et_576470', 'et_601753', 'et_619958', 'et_629959_2', 'et_664304_2', 'et_677251', 'et_712871', 'et_848643', 'et_862513', 'et_963607']
Número de sujetos: 44


In [56]:
import shutil

#shutil.rmtree('Features')
print('Carpeta Features eliminada.')

Carpeta Features eliminada.


## 1. Información Bottom-Up

FEATURES:
- saliencia_media: media del mapa de saliencia en los pixeles de la grilla
- saliencia_mediana: mediana del mapa de saliencia en los pixeles de la grilla

In [ ]:
# ── Mapa de saliencia CORREGIDO ──────────────
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    with open(f'../Datasets/HSEM/human_scanpaths/{sujeto}_scanpaths.json') as f:
        scanpaths = json.load(f)

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')
        
        if imagen not in scanpaths:
            print(f'    ⚠️  Sin scanpath para {imagen}, saltando...')
            continue

        mapa_sal = ml.cargar_mapa_saliencia(imagen, como_grilla=False)
        mapas    = ml.cargar_mapas_humanos(sujeto, imagen)

        scanpath   = scanpaths[imagen]
        scr_width  = scanpath['screen_width']
        scr_height = scanpath['screen_height']
        xs_pantalla = scanpath['X']
        ys_pantalla = scanpath['Y']

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        # CORRECCIÓN: Inicializamos 'fijaciones' como lista vacía para llenarla en el loop
        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [], 
        })
        
        # Limpiamos fijaciones previas si el JSON ya existía para evitar duplicados o desfases
        data['fijaciones'] = []

        for i, (x_p, y_p) in enumerate(zip(xs_pantalla, ys_pantalla)):

            # 1. Coordenadas en espacio del mapa (1024x768)
            x_img = min(int(x_p * (mapa_sal.shape[1] / scr_width)),  mapa_sal.shape[1] - 1)
            y_img = min(int(y_p * (mapa_sal.shape[0] / scr_height)), mapa_sal.shape[0] - 1)

            # 2. Determinar la celda real
            grid_x_real = x_img // 32
            grid_y_real = y_img // 32

            # 3. Definir el bloque 32x32
            y_ini = grid_y_real * 32
            x_ini = grid_x_real * 32
            y_fin = min(y_ini + 32, mapa_sal.shape[0])
            x_fin = min(x_ini + 32, mapa_sal.shape[1])
            
            bloque = mapa_sal[y_ini:y_fin, x_ini:x_fin]

            # 4. CORRECCIÓN: Usamos .append() en lugar de .update() por índice
            data['fijaciones'].append({
                'fixation':          i,
                'fix_y':             int(grid_y_real),
                'fix_x':             int(grid_x_real),
                'saliencia_media':   float(np.mean(bloque)) if bloque.size > 0 else 0.0,
                'saliencia_mediana': float(np.median(bloque)) if bloque.size > 0 else 0.0,
                'saliencia_pixel':   float(mapa_sal[y_img, x_img]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

In [ ]:
# ── Mapa de saliencia ORIGINAL ──────────────────────────────────────────────────────
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    with open(f'../Datasets/HSEM/human_scanpaths/{sujeto}_scanpaths.json') as f:
        scanpaths = json.load(f)

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')
        
        if imagen not in scanpaths:
            print(f'    ⚠️  Sin scanpath para {imagen}, saltando...')
            continue

        mapa_sal = ml.cargar_mapa_saliencia(imagen, como_grilla=False)
        mapas    = ml.cargar_mapas_humanos(sujeto, imagen)

        scanpath   = scanpaths[imagen]
        scr_width  = scanpath['screen_width']
        scr_height = scanpath['screen_height']
        xs_pantalla = scanpath['X']
        ys_pantalla = scanpath['Y']

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        for i, (y, x, x_p, y_p) in enumerate(zip(
                mapas['fixations_y'], mapas['fixations_x'],
                xs_pantalla, ys_pantalla)):

            # Coordenadas en espacio del mapa
            x_img = min(int(x_p * (mapa_sal.shape[1] / scr_width)),  mapa_sal.shape[1] - 1)
            y_img = min(int(y_p * (mapa_sal.shape[0] / scr_height)), mapa_sal.shape[0] - 1)

            y_ini = y * 32
            x_ini = x * 32
            y_fin = min(y_ini + 32, mapa_sal.shape[0])
            x_fin = min(x_ini + 32, mapa_sal.shape[1])
            bloque = mapa_sal[y_ini:y_fin, x_ini:x_fin]

            data['fijaciones'][i].update({
                'fixation':          i,
                'fix_y':             int(y),
                'fix_x':             int(x),
                'saliencia_media_orig':   float(np.mean(bloque)),
                'saliencia_mediana_orig': float(np.median(bloque)),
                'saliencia_pixel_orig':   float(mapa_sal[y_img, x_img]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
  Procesando imagen: cmp_beach_002_person_002.jpg
  Procesando imagen: cmp_beach_003_bird_001.jpg
  Procesando imagen: cmp_building_003_person_001.jpg
  Procesando imagen: cmp_building_006_person_003.jpg
  Procesando imagen: cmp_building_007_person_004.jpg
  Procesando imagen: cmp_building_010_cat_002.jpg
  Procesando imagen: cmp_building_014_person_007.jpg
  Procesando imagen: cmp_building_015_person_008.jpg
  Procesando imagen: cmp_building_018_cat_005.jpg
  Procesando imagen: cmp_building_020_cat_007.jpg
  Procesando imagen: cmp_building_021_cat_008.jpg
  Procesando imagen: cmp_building_022_cat_009.jpg
  Procesando imagen: cmp_building_023_cat_010.jpg
  Procesando imagen: cmp_building_030_person_016.jpg
  Procesando imagen: cmp_building_032_cat_011.jpg
  Procesando imagen: cmp_building_033_cat_012.jpg
  Procesando imagen: cmp_building_039_dog_007.jpg
  Procesando imagen: 

## 2. Target Similarity

FEATURES:
- similitud_media: media del mapa de similitud en los pixeles de la grilla
- similitud_mediana: mediana del mapa de similitud en los pixeles de la grilla

In [58]:
# ── Mapa de similitud ──────────────────────────────────────────────────────
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    with open(f'../Datasets/HSEM/human_scanpaths/{sujeto}_scanpaths.json') as f:
        scanpaths = json.load(f)

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')

        if imagen not in scanpaths:
            print(f'    ⚠️  Sin scanpath para {imagen}, saltando...')
            continue

        mapas  = ml.cargar_mapas_humanos(sujeto, imagen)
        target = mapas['target_stim']

        if not ml.mapa_similitud_existe(imagen, target):
            print(f'    ⚠️  Sin mapa de similitud para {imagen} / {target}')
            continue

        mapa_sim = ml.cargar_mapa_similitud(imagen, target, como_grilla=False)

        scanpath    = scanpaths[imagen]
        scr_width   = scanpath['screen_width']
        scr_height  = scanpath['screen_height']
        xs_pantalla = scanpath['X']
        ys_pantalla = scanpath['Y']

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  target,
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        for i, (y, x, x_p, y_p) in enumerate(zip(
                mapas['fixations_y'], mapas['fixations_x'],
                xs_pantalla, ys_pantalla)):

            x_img = min(int(x_p * (mapa_sim.shape[1] / scr_width)),  mapa_sim.shape[1] - 1)
            y_img = min(int(y_p * (mapa_sim.shape[0] / scr_height)), mapa_sim.shape[0] - 1)

            y_ini = y * 32
            x_ini = x * 32
            y_fin = min(y_ini + 32, mapa_sim.shape[0])
            x_fin = min(x_ini + 32, mapa_sim.shape[1])
            bloque_sim = mapa_sim[y_ini:y_fin, x_ini:x_fin]

            data['fijaciones'][i].update({
                'similitud_media':   float(np.mean(bloque_sim)),
                'similitud_mediana': float(np.median(bloque_sim)),
                'similitud_pixel':   float(mapa_sim[y_img, x_img]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
  Procesando imagen: cmp_beach_002_person_002.jpg
  Procesando imagen: cmp_beach_003_bird_001.jpg
  Procesando imagen: cmp_building_003_person_001.jpg
  Procesando imagen: cmp_building_006_person_003.jpg
  Procesando imagen: cmp_building_007_person_004.jpg
  Procesando imagen: cmp_building_010_cat_002.jpg
  Procesando imagen: cmp_building_014_person_007.jpg
  Procesando imagen: cmp_building_015_person_008.jpg
  Procesando imagen: cmp_building_018_cat_005.jpg
  Procesando imagen: cmp_building_020_cat_007.jpg
  Procesando imagen: cmp_building_021_cat_008.jpg
  Procesando imagen: cmp_building_022_cat_009.jpg
  Procesando imagen: cmp_building_023_cat_010.jpg
  Procesando imagen: cmp_building_030_person_016.jpg
  Procesando imagen: cmp_building_032_cat_011.jpg
  Procesando imagen: cmp_building_033_cat_012.jpg
  Procesando imagen: cmp_building_039_dog_007.jpg
  Procesando imagen: 

## 3. Evidencia Visual

In [59]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    print(f'Procesando sujeto: {sujeto}, número de imágenes: {len(imagenes)}')

    for imagen in imagenes:
        print(f'  Procesando imagen: {imagen}')
        
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # ve_at_fix: valor de la evidencia visual acumulada en la celda fijada
        ve_at_fix = ml.valor_en_fijacion(mapas['visual_evidence'], mapas)

        for i, (y, x) in enumerate(zip(mapas['fixations_y'], mapas['fixations_x'])):
            data['fijaciones'][i].update({
                'fixation':   i,
                'fix_y':      int(y),
                'fix_x':      int(x),
                'evidencia_visual':  float(ve_at_fix[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

Procesando sujeto: S101, número de imágenes: 102
  Procesando imagen: cmp_africansavanna_003_wildanimals_003.jpg
  Procesando imagen: cmp_beach_002_person_002.jpg
  Procesando imagen: cmp_beach_003_bird_001.jpg
  Procesando imagen: cmp_building_003_person_001.jpg
  Procesando imagen: cmp_building_006_person_003.jpg
  Procesando imagen: cmp_building_007_person_004.jpg
  Procesando imagen: cmp_building_010_cat_002.jpg
  Procesando imagen: cmp_building_014_person_007.jpg
  Procesando imagen: cmp_building_015_person_008.jpg
  Procesando imagen: cmp_building_018_cat_005.jpg
  Procesando imagen: cmp_building_020_cat_007.jpg
  Procesando imagen: cmp_building_021_cat_008.jpg
  Procesando imagen: cmp_building_022_cat_009.jpg
  Procesando imagen: cmp_building_023_cat_010.jpg
  Procesando imagen: cmp_building_030_person_016.jpg
  Procesando imagen: cmp_building_032_cat_011.jpg
  Procesando imagen: cmp_building_033_cat_012.jpg
  Procesando imagen: cmp_building_039_dog_007.jpg
  Procesando imagen: 

## 4. Competencia Foveal-Periferica

Guardo la varianza, la media y el máximo de la competencia foveal-periferica

In [60]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        competencia_var    = np.var(mapas['visual_evidence'],  axis=(1, 2))
        competencia_media  = np.mean(mapas['visual_evidence'], axis=(1, 2))
        competencia_max    = np.max(mapas['visual_evidence'],  axis=(1, 2))

        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'competencia_fp_var':   float(competencia_var[i]),
                'competencia_fp_media': float(competencia_media[i]),
                'competencia_fp_max':   float(competencia_max[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 5. Probabilidad Posterior

In [61]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Valor del posterior en la celda fijada — escalar por fijación
        posterior_at_fix = ml.valor_en_fijacion(mapas['posterior'], mapas)

        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'posterior': float(posterior_at_fix[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 6. Entropía del Prior

El mapa de saliencia (DeepGaze II) no es una distribución de probabilidad: sus valores están en [0,1] pero no suman 1. Es una función de puntuación que indica relevancia visual relativa de cada celda.
El modelo lo convierte en distribución en dos pasos: primero normaliza por el máximo, luego aplica una transformación afín que garantiza peso mínimo a cada celda. El posterior resultante en t=0 sí es una distribución válida (suma 1) y representa la creencia inicial real del modelo.
Se guardan dos versiones:

entropia_prior_saliencia: entropía del mapa de saliencia crudo normalizado manualmente. Mide la complejidad visual de la escena.
entropia_prior_modelo: entropía del posterior[0], es decir el prior procesado que realmente usa el modelo. Mide la incertidumbre inicial del proceso de búsqueda.

Ambas son constantes a lo largo del trial (el prior no cambia entre fijaciones).

In [62]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas    = ml.cargar_mapas_humanos(sujeto, imagen)
        mapa_sal = ml.cargar_mapa_saliencia(imagen, como_grilla=True)  # (24, 32)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Opción A: entropía de la saliencia cruda normalizada
        prior_sal  = mapa_sal / mapa_sal.sum()
        entropia_a = float(-np.sum(prior_sal * np.log(prior_sal + 1e-12)))

        # Opción B: entropía del prior real del modelo (posterior en fijación 0)
        prior_mod  = mapas['posterior'][0]
        prior_mod  = prior_mod / prior_mod.sum()
        entropia_b = float(-np.sum(prior_mod * np.log(prior_mod + 1e-12)))

        # Es una feature de la imagen, no de cada fijación — igual valor en todas
        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'entropia_prior_saliencia': entropia_a,
                'entropia_prior_modelo':    entropia_b,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 7. Entropía del Posterior

In [63]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # Entropía de Shannon del posterior en cada fijación
        # entropy_map ya tiene -p·log(p) celda a celda — shape (n_fix, 24, 32)
        # Su suma sobre el mapa da la entropía escalar
        entropia_posterior = ml.entropia_escalar(mapas)  # shape (n_fix,)

        for i in range(len(mapas['fixations_y'])):
            data['fijaciones'][i].update({
                'entropia_posterior': float(entropia_posterior[i]),
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 8. Ganancia de la información

def kl_divergencia_consecutiva(data: dict) -> np.ndarray:
    """
    Divergencia KL entre posteriors consecutivos: KL(P_t || P_{t-1}).

    Mide cuánta información aportó cada fijación al actualizar la creencia.
    Análogo al elm_info_gained del modelo, pero calculado sobre los posteriors reales.

    Shape: (n_fix - 1,)  — la fijación 0 no tiene referencia anterior.
    """
    post = data['posterior']   # (n_fix, 24, 32)
    eps = 1e-12
    return np.sum(post[1:] * np.log((post[1:] + eps) / (post[:-1] + eps)), axis=(1, 2))

In [64]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        # KL(P_t || P_{t-1}) — shape (n_fix - 1,)
        kl = ml.kl_divergencia_consecutiva(mapas)

        for i in range(len(mapas['fixations_y'])):
            # La fijación 0 no tiene referencia anterior
            kl_val = float(kl[i - 1]) if i > 0 else None

            data['fijaciones'][i].update({
                'ganancia_informacion': kl_val,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 9. Gap de Entropía con el Óptimo

In [65]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)
        fix_ys  = mapas['fixations_y']
        fix_xs  = mapas['fixations_x']
        n_fix   = len(fix_ys)

        for i in range(n_fix):

            optimo = float(eig_map[i].max())

            if i < n_fix - 1:
                y_sig            = fix_ys[i + 1]
                x_sig            = fix_xs[i + 1]
                eig_en_siguiente = float(eig_map[i, y_sig, x_sig])
                gap              = optimo - eig_en_siguiente
            else:
                eig_en_siguiente = None
                gap              = None

            data['fijaciones'][i].update({
                'eig_optimo':      optimo,
                'eig_en_fijacion': eig_en_siguiente,
                'gap_entropia':    gap,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 10. Distancia con decisión del modelo

In [66]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)
        fix_ys  = mapas['fixations_y']
        fix_xs  = mapas['fixations_x']
        n_fix   = len(fix_ys)

        for i in range(n_fix):

            coords_optimo  = np.unravel_index(eig_map[i].argmax(), eig_map[i].shape)
            y_modelo, x_modelo = coords_optimo

            if i < n_fix - 1:
                y_sig            = fix_ys[i + 1]
                x_sig            = fix_xs[i + 1]
                distancia_grilla = float(np.sqrt((y_sig - y_modelo)**2 + (x_sig - x_modelo)**2))
                distancia_pixels = distancia_grilla * 32
            else:
                distancia_grilla = None
                distancia_pixels = None

            data['fijaciones'][i].update({
                'modelo_fix_y':     int(y_modelo),
                'modelo_fix_x':     int(x_modelo),
                'distancia_grilla': distancia_grilla,
                'distancia_pixels': distancia_pixels,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 11. Competencia en Matriz de Entropias (ratio)

In [67]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)

        for i in range(len(mapas['fixations_y'])):

            # Aplanar y ordenar de mayor a menor
            valores = np.sort(eig_map[i].flatten())[::-1]

            mayor        = float(valores[0])
            segundo_mayor = float(valores[1])

            # Evitar división por cero
            ratio = float(segundo_mayor / mayor) if mayor > 0 else None

            data['fijaciones'][i].update({
                'competencia_ratio_mayor':        mayor,
                'competencia_ratio_segundo_mayor': segundo_mayor,
                'competencia_ratio':              ratio,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)

## 12. Competencia en matriz de entropias (densidad)  

In [68]:
for sujeto in sujetos:
    imagenes = ml.listar_imagenes(sujeto)
    for imagen in imagenes:
        mapas = ml.cargar_mapas_humanos(sujeto, imagen)

        imagen_stem = imagen.replace('.jpg', '')
        output_dir  = f'Features/{imagen_stem}/{sujeto}'
        os.makedirs(output_dir, exist_ok=True)
        json_path   = f'{output_dir}/features.json'

        data = cargar_o_crear_json(json_path, defaults={
            'imagen':       imagen,
            'sujeto':       sujeto,
            'target_stim':  mapas['target_stim'],
            'target_found': mapas['target_found'],
            'memory_set':   mapas['memory_set'],
            'fijaciones':   [{} for _ in range(len(mapas['fixations_y']))],
        })

        eig_map = mapas['expected_ig_map']   # (n_fix, 24, 32)

        for i in range(len(mapas['fixations_y'])):

            mayor   = float(eig_map[i].max())
            umbral  = mayor * 0.95

            # Cantidad de celdas dentro del 5% del máximo
            densidad = int(np.sum(eig_map[i] >= umbral))

            # Versión normalizada: proporción sobre el total de celdas (24*32=768)
            densidad_norm = float(densidad / eig_map[i].size)

            data['fijaciones'][i].update({
                'competencia_densidad':      densidad,
                'competencia_densidad_norm': densidad_norm,
            })

        with open(json_path, 'w') as f:
            json.dump(data, f, indent=2)